# Optional Part 9: Manual Loop to PyTorch Lightning

This notebook is an optional translation after Part 8B. It does not introduce a second training recipe. It keeps the ordinary PyTorch model and moves orchestration into:

- `training_step`;
- `validation_step`;
- `configure_optimizers`;
- `Trainer`;
- `ModelCheckpoint`.

The acceptance tests are concrete: manual and Lightning losses must match on the same batch, output shapes must match, and a reloaded Lightning checkpoint must preserve logits.


In [ ]:
# Imports and deterministic optional lab state
import tempfile
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import lightning.pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint

SEED = 909
pl.seed_everything(SEED, workers=True)
torch.set_num_threads(1)


In [ ]:
# A small fixed data contract for orchestration parity
vocab_size, sequence_length = 17, 8
generator = torch.Generator().manual_seed(SEED)
examples = torch.randint(
    1, vocab_size, (96, sequence_length + 1), generator=generator
)
inputs, labels = examples[:, :-1], examples[:, 1:]

class TinyDataModule(pl.LightningDataModule):
    def setup(self, stage=None):
        self.train_data = TensorDataset(inputs[:80], labels[:80])
        self.validation_data = TensorDataset(inputs[80:], labels[80:])

    def train_dataloader(self):
        return DataLoader(self.train_data, batch_size=16, shuffle=False)

    def val_dataloader(self):
        return DataLoader(self.validation_data, batch_size=16, shuffle=False)

data = TinyDataModule()
data.setup("fit")
sample_batch = next(iter(data.train_dataloader()))
print("input shape:", tuple(sample_batch[0].shape))
print("label shape:", tuple(sample_batch[1].shape))


In [ ]:
# The model remains plain PyTorch
class TinyCausalTransformer(nn.Module):
    def __init__(self, vocab_size, max_length, d_model=24):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_length, d_model)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=4,
            dim_feedforward=48,
            dropout=0.0,
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(
            layer, num_layers=1, enable_nested_tensor=False
        )
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, input_ids):
        length = input_ids.size(1)
        positions = torch.arange(length, device=input_ids.device)
        hidden = self.token_embedding(input_ids) + self.position_embedding(positions)
        causal = torch.triu(
            torch.ones(length, length, dtype=torch.bool, device=input_ids.device),
            diagonal=1,
        )
        hidden = self.transformer(hidden, mask=causal, is_causal=True)
        return self.lm_head(hidden)

def causal_loss(model, batch):
    input_ids, targets = batch
    logits = model(input_ids)
    loss = F.cross_entropy(
        logits.flatten(0, 1), targets.flatten()
    )
    return logits, loss


## Lightning wraps policy; it does not own Transformer math

`training_step` and `validation_step` call the same `causal_loss`. Lightning owns mode changes, gradient clearing, backward, optimizer stepping, and callback timing. Keeping the core model plain PyTorch makes the boundary inspectable.


In [ ]:
# Lightning hooks are a thin orchestration layer
class LitCausalTransformer(pl.LightningModule):
    def __init__(self, model, learning_rate=3e-3):
        super().__init__()
        self.model = model
        self.learning_rate = learning_rate

    def forward(self, input_ids):
        return self.model(input_ids)

    def training_step(self, batch, batch_idx):
        _, loss = causal_loss(self.model, batch)
        self.log("train_loss", loss, on_step=True, on_epoch=False)
        return loss

    def validation_step(self, batch, batch_idx):
        _, loss = causal_loss(self.model, batch)
        self.log("validation_loss", loss, on_epoch=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.AdamW(
            self.model.parameters(), lr=self.learning_rate
        )


In [ ]:
# Shape and loss parity before Trainer owns any updates
torch.manual_seed(SEED)
core_model = TinyCausalTransformer(vocab_size, sequence_length)
lightning_model = LitCausalTransformer(core_model)
manual_logits, manual_loss = causal_loss(core_model, sample_batch)
lightning_loss = lightning_model.training_step(sample_batch, 0)

print("logit shape:", tuple(manual_logits.shape))
print(f"manual loss:    {manual_loss.item():.8f}")
print(f"Lightning loss: {lightning_loss.item():.8f}")
assert manual_logits.shape == (16, sequence_length, vocab_size)
assert torch.allclose(manual_loss, lightning_loss)


In [ ]:
# Trainer and callback parity, contained in a temporary directory
with tempfile.TemporaryDirectory(prefix="transformer-lightning-") as temp_dir:
    checkpoint_callback = ModelCheckpoint(
        dirpath=temp_dir,
        filename="tiny-{epoch:02d}",
        save_top_k=1,
        monitor="validation_loss",
        mode="min",
    )
    trainer = pl.Trainer(
        max_epochs=1,
        accelerator="cpu",
        devices=1,
        logger=False,
        enable_progress_bar=False,
        enable_model_summary=False,
        deterministic=True,
        limit_train_batches=3,
        limit_val_batches=1,
        callbacks=[checkpoint_callback],
    )
    trainer.fit(lightning_model, datamodule=data)
    checkpoint_path = Path(checkpoint_callback.best_model_path)
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)

    restored_core = TinyCausalTransformer(vocab_size, sequence_length)
    restored = LitCausalTransformer(restored_core)
    restored.load_state_dict(checkpoint["state_dict"])
    lightning_model.eval()
    restored.eval()
    with torch.no_grad():
        expected_logits = lightning_model(sample_batch[0])
        restored_logits = restored(sample_batch[0])
    checkpoint_delta = (
        expected_logits - restored_logits
    ).abs().max().item()
    print("checkpoint file:", checkpoint_path.name)
    print(f"reload logit delta: {checkpoint_delta:.8f}")
    assert checkpoint_delta < 1e-7

print("temporary Trainer artifacts removed:", not Path(temp_dir).exists())


## Optional Part 9 checkpoint

| Manual lifecycle | Lightning owner |
|---|---|
| batch loop | `Trainer.fit` |
| forward + loss | `training_step` / `validation_step` |
| optimizer construction | `configure_optimizers` |
| best-checkpoint save | `ModelCheckpoint` |

The parity checks passed on the same shapes, loss function, model weights, and checkpointed logits. Lightning reduced orchestration code; it did not replace the causal objective or the artifact reasoning learned in Part 8B.

**Optional boundary:** distributed strategies, mixed precision, and experiment loggers belong after this parity is understood.
